# Prerequisites

## Load Bronze data to dataframe

In [0]:
%sql
use catalog ecommerce;
create schema if not exists silver;


In [0]:
payments_df = spark.read.table("bronze.payments")
display(payments_df.count())
payments_df.limit(5).display()

In [0]:
payments_df.printSchema()

# Transformations

## Drop Duplicates, payment_id IS NOT NULL , payment_amount > 0

In [0]:
from pyspark.sql.functions import *
payments_df = payments_df.filter(col("payment_id").isNotNull()).filter(col("payment_amount")>0)
payments_df = payments_df.dropDuplicates()
display(payments_df.count())
payments_df.limit(5).display()

## Standardize payment method.

In [0]:
valid_payment_method = ["NET_BANKING", "UPI", "COD", "WALLET", "CARD"]
payments_df = payments_df.withColumn(
    "payment_method", upper(trim(col("payment_method")))
).filter(col("payment_method").isin(valid_payment_method))
display(payments_df.count())
payments_df.limit(5).display()

## Standardize payment status.

In [0]:
valid_payment_status = ["SUCCESS", "FAILED", "PENDING", "REFUNDED"]
payments_df = payments_df.withColumn("payment_status",upper(trim(col("payment_status")))).filter(
    col("payment_status").isin(valid_payment_status)
)
display(payments_df.count())
payments_df.limit(5).display()

## Identify failed payments , successful payments, Identify refunds.

In [0]:
failed_payments_df = payments_df.filter(upper(trim(col("payment_status")))=="FAILED")
successful_payments_df = payments_df.filter(upper(trim(col("payment_status")))=="SUCCESS")
refunds_paymets_df = payments_df.filter(upper(trim(col("payment_status")))=="REFUNDED")
display(failed_payments_df.count())
failed_payments_df.limit(5).display()
display(successful_payments_df.count())
successful_payments_df.limit(5).display()
display(refunds_paymets_df.count())
refunds_paymets_df.limit(5).display()

# Create Silver table

## Write to silver schema

In [0]:
%sql
--drop table if exists ecommerce.silver.payments

In [0]:
payments_df.write.format("delta").mode("overwrite").saveAsTable("ecommerce.silver.payments")

## Read silver.payments table

In [0]:
spark.read.table("ecommerce.silver.payments").display()